# Gradient Boosting Model - Home Credit Default Risk

This notebook follows the project specification for predicting credit default risk with the Home Credit Default Risk dataset. The main metric is ROC-AUC, with supporting evaluation through precision, recall, F1-score, ROC curve, precision-recall curve, confusion matrix, feature importance, and a Kaggle submission file.

If the Kaggle CSV files are not present in `data/raw`, the notebook runs the included synthetic demo dataset so the full workflow can still be tested.

## 1. Setup

Place the Kaggle files here before running on real data:

- `data/raw/application_train.csv`
- `data/raw/application_test.csv`

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import SVG, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

OUTPUTS = PROJECT_ROOT / 'outputs'
FIGURES = PROJECT_ROOT / 'reports' / 'figures'
RAW = PROJECT_ROOT / 'data' / 'raw'

PROJECT_ROOT

## 2. Run the Pipeline

The notebook reuses the tested project pipeline so the notebook, report, figures, and Kaggle submission are generated from the same implementation.

In [ ]:
import credit_default_pipeline as pipeline

pipeline.main()

## 3. Run Metadata

This confirms whether the notebook used real Kaggle data or synthetic demo data.

In [ ]:
metadata = json.loads((OUTPUTS / 'run_metadata.json').read_text())
metadata

## 4. Missing Values and Target Imbalance

Credit datasets often contain many missing values. The target is also imbalanced, so AUC is more informative than raw accuracy.

In [ ]:
missing = pd.read_csv(OUTPUTS / 'missing_value_summary.csv')
missing.head(15)

In [ ]:
target_dist = pd.read_csv(OUTPUTS / 'target_distribution.csv')
target_dist

In [ ]:
display(SVG(filename=str(FIGURES / 'target_distribution.svg')))

## 5. Correlation Review

The heatmap focuses on selected numeric variables, including external score variables and engineered credit ratios.

In [ ]:
display(SVG(filename=str(FIGURES / 'correlation_heatmap.svg')))

## 6. Model Comparison

The project compares a baseline logistic model with a gradient boosting fallback. If `scikit-learn`, `LightGBM`, `XGBoost`, or `CatBoost` are installed, the pipeline automatically includes them too.

In [ ]:
comparison = pd.read_csv(OUTPUTS / 'model_comparison.csv')
comparison

## 7. Out-of-Sample Evaluation

The main metric is ROC-AUC. Precision, recall, F1-score, and the confusion matrix use a validation-selected threshold because default risk is an imbalanced classification problem.

In [ ]:
display(SVG(filename=str(FIGURES / 'roc_curve.svg')))
display(SVG(filename=str(FIGURES / 'precision_recall_curve.svg')))
display(SVG(filename=str(FIGURES / 'confusion_matrix.svg')))

## 8. Feature Importance and SHAP Proxy

When SHAP is installed, this section can be extended to a true SHAP summary plot. In the self-contained fallback environment, the project writes a permutation-importance proxy with the same purpose: explaining which variables moved the model most.

In [ ]:
importance = pd.read_csv(OUTPUTS / 'feature_importance.csv')
importance.head(20)

In [ ]:
display(SVG(filename=str(FIGURES / 'feature_importance.svg')))
display(SVG(filename=str(FIGURES / 'shap_summary_or_proxy.svg')))

## 9. Submission File

With real Kaggle test data, the pipeline writes `outputs/kaggle_submission.csv`. In demo mode, it writes `outputs/demo_submission.csv`.

In [ ]:
submission_candidates = sorted(OUTPUTS.glob('*submission.csv'))
submission_candidates

In [ ]:
if submission_candidates:
    display(pd.read_csv(submission_candidates[-1]).head())

## 10. Final Report

The generated report summarizes the main results in a submission-friendly format.

In [ ]:
print((OUTPUTS / 'credit_default_report.md').read_text())